# GenerativeAURORA — NSRDB Data Downloader

This notebook is the Jupyter version of `download_nsrdb.py`. It follows the same workflow while separating documentation, configuration, functions, execution, and verification into readable cells.

**Workflow:** NSRDB API → 5 sites × 5 years → 25 hourly CSV files → local clearness-index/sky-regime calculation → checkpointed download → `merge_nsrdb.py`.

**Security:** keep the NREL API key out of shared notebooks and repositories. The credential cell below uses environment variables.

## 1. Import required packages

`requests` handles the NREL API request, `pandas` processes the returned CSV data, and the standard-library modules handle files, JSON checkpointing, and request timing.

In [1]:
!pip -q install requests pandas

import os
import time
import json
from io import StringIO

import requests
import pandas as pd

print('Packages imported successfully.')

Packages imported successfully.


## 2. User configuration

Load credentials from environment variables rather than hard-coding them. If necessary for a private run, replace the placeholders only in your local environment.

In [3]:
from google.colab import userdata

API_KEY = userdata.get('NREL_API_KEY')
EMAIL = userdata.get('NREL_EMAIL')

if not API_KEY:
    raise ValueError('NREL_API_KEY is missing. Add it in Colab Secrets and enable notebook access.')

if not EMAIL:
    raise ValueError('NREL_EMAIL is missing. Add it in Colab Secrets and enable notebook access.')

print('Credentials loaded successfully.')

Credentials loaded successfully.


## 3. Define study sites and download period

The dataset uses five representative climate locations and five years of hourly data (2018–2022), giving 25 site-year downloads.

In [4]:
SITES = [
    {'name': 'Phoenix_AZ',    'lat': 33.4484,  'lon': -112.0740, 'climate': 'hot_desert'},
    {'name': 'LosAngeles_CA', 'lat': 34.0522,  'lon': -118.2437, 'climate': 'mediterranean'},
    {'name': 'Denver_CO',     'lat': 39.7392,  'lon': -104.9903, 'climate': 'semi_arid'},
    {'name': 'Miami_FL',      'lat': 25.7617,  'lon':  -80.1918, 'climate': 'subtropical'},
    {'name': 'Seattle_WA',    'lat': 47.6062,  'lon': -122.3321, 'climate': 'oceanic'},
]

YEARS = [2018, 2019, 2020, 2021, 2022]

print(f'Sites: {len(SITES)}')
print(f'Years: {YEARS}')
print(f'Expected downloads: {len(SITES) * len(YEARS)}')

Sites: 5
Years: [2018, 2019, 2020, 2021, 2022]
Expected downloads: 25


## 4. Configure the current GOES Aggregated PSM v4 API

The API request uses the same attributes and parameters as the original script: 60-minute interval, local-time output (`utc=false`), leap-day data, and the specified NSRDB variables.

In [5]:
ATTRIBUTES = ','.join([
    'ghi',
    'dhi',
    'dni',
    'wind_speed',
    'air_temperature',
    'relative_humidity',
    'surface_pressure',
    'cloud_type',
    'solar_zenith_angle',
    'clearsky_ghi',
])

INTERVAL = '60'
UTC = 'false'
LEAP_DAY = 'true'
FULL_NAME = 'Researcher'      #input your full name
AFFILIATION = 'GAWIREA'       #input your institution/organization
REASON = 'Academic Research'  #input your reason
MAILING_LIST = 'false'

BASE_URL = (
    'https://developer.nlr.gov/api/nsrdb/v2/solar/'
    'nsrdb-GOES-aggregated-v4-0-0-download.csv'
)

OUTPUT_DIR = 'nsrdb_data'
CHECKPOINT_FILE = 'download_checkpoint.json'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'API endpoint: {BASE_URL}')
print(f'Interval: {INTERVAL} minutes')
print(f'UTC: {UTC}')

API endpoint: https://developer.nlr.gov/api/nsrdb/v2/solar/nsrdb-GOES-aggregated-v4-0-0-download.csv
Interval: 60 minutes
UTC: false


## 5. Checkpoint functions

`load_checkpoint()` reads previously completed site-year combinations. `save_checkpoint()` writes them after every successful download so an interrupted run can be resumed without repeating completed downloads.

In [6]:
def load_checkpoint():
    """Load completed (site, year) combinations."""
    if not os.path.exists(CHECKPOINT_FILE):
        return set()
    with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)
    completed = {tuple(item) for item in data.get('completed', [])}
    print(f'Checkpoint loaded: {len(completed)} completed files.')
    return completed

def save_checkpoint(completed):
    """Save completed (site, year) combinations."""
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'completed': [list(item) for item in sorted(completed)]}, f, indent=2)

## 6. Test one NSRDB request before starting all 25 downloads

This diagnostic cell is important. It tests authentication and the API endpoint for one site-year only. It does not save the file yet.
Possible outcomes:
- 200 → API request works; proceed to the full download.
- 403/401 → API key/authentication problem.
- 400 → one or more request parameters are invalid.
- 429 → rate limit exceeded.
- 5xx → server-side/API issue.

In [7]:
test_site = SITES[0]
test_year = YEARS[0]

test_params = {
    'wkt': f"POINT({test_site['lon']} {test_site['lat']})",
    'attributes': ATTRIBUTES,
    'names': str(test_year),
    'utc': UTC,
    'leap_day': LEAP_DAY,
    'interval': INTERVAL,
    'full_name': FULL_NAME,
    'email': EMAIL,
    'affiliation': AFFILIATION,
    'mailing_list': MAILING_LIST,
    'reason': REASON,
    'api_key': API_KEY,
}

response = requests.get(BASE_URL, params=test_params, timeout=120)

print(f'HTTP status: {response.status_code}')
print(f'Content type: {response.headers.get("content-type", "unknown")}')
print('\nResponse preview:')
print(response.text[:1200])

if response.status_code != 200:
    raise RuntimeError(
        f'NSRDB API test failed with HTTP {response.status_code}. '
        'See the response preview above.'
    )

print('\nAPI test PASSED.')

HTTP status: 200
Content type: text/csv

Response preview:
Source,Location ID,City,State,Country,Latitude,Longitude,Time Zone,Elevation,Local Time Zone,Clearsky DHI Units,Clearsky DNI Units,Clearsky GHI Units,Dew Point Units,DHI Units,DNI Units,GHI Units,Solar Zenith Angle Units,Temperature Units,Pressure Units,Relative Humidity Units,Precipitable Water Units,Wind Direction Units,Wind Speed Units,Cloud Type -15,Cloud Type 0,Cloud Type 1,Cloud Type 2,Cloud Type 3,Cloud Type 4,Cloud Type 5,Cloud Type 6,Cloud Type 7,Cloud Type 8,Cloud Type 9,Cloud Type 10,Cloud Type 11,Cloud Type 12,Fill Flag 0,Fill Flag 1,Fill Flag 2,Fill Flag 3,Fill Flag 4,Fill Flag 5,Surface Albedo Units,Version
NSRDB,323705,-,-,-,33.45,-112.06,-7,334,-7,w/m2,w/m2,w/m2,c,w/m2,w/m2,w/m2,Degree,c,mbar,%,cm,Degrees,m/s,N/A,Clear,Probably Clear,Fog,Water,Super-Cooled Water,Mixed,Opaque Ice,Cirrus,Overlapping,Overshooting,Unknown,Dust,Smoke,N/A,Missing Image,Low Irradiance,Exceeds Clearsky,Missing CLoud Properties,Rayleigh 

## 7. Download and process one site-year

For each site-year, the function requests the NSRDB CSV, parses the returned data, adds site metadata, calculates `clearness_index = GHI / Clearsky GHI` (clipped to [0, 1]), assigns the three sky regimes, and saves the result as a CSV.

In [8]:
def download_site_year(site, year):
    """Download, process, and save one site-year dataset."""
    filename = os.path.join(OUTPUT_DIR, f"{site['name']}_{year}.csv")

    params = {
        'wkt': f"POINT({site['lon']} {site['lat']})",
        'attributes': ATTRIBUTES,
        'names': str(year),
        'utc': UTC,
        'leap_day': LEAP_DAY,
        'interval': INTERVAL,
        'full_name': FULL_NAME,
        'email': EMAIL,
        'affiliation': AFFILIATION,
        'mailing_list': MAILING_LIST,
        'reason': REASON,
        'api_key': API_KEY,
    }

    print(f'Downloading {site["name"]} {year}...')

    try:
        response = requests.get(BASE_URL, params=params, timeout=120)

        if not response.ok:
            print(f'HTTP {response.status_code}: {response.reason}')
            print('NLR response:')
            print(response.text[:1500])
            return False

        df = pd.read_csv(StringIO(response.text), skiprows=2)

        df['site'] = site['name']
        df['climate'] = site['climate']
        df['lat'] = site['lat']
        df['lon'] = site['lon']

        required = {'GHI', 'Clearsky GHI'}
        if not required.issubset(df.columns):
            print('ERROR: Required irradiance columns were not returned.')
            print(f'Columns returned: {list(df.columns)}')
            return False

        df['clearness_index'] = (
            df['GHI'] / df['Clearsky GHI'].replace(0, float('nan'))
        ).clip(0, 1).fillna(0)

        df['sky_regime'] = 'Overcast'
        df.loc[df['clearness_index'] > 0.10, 'sky_regime'] = 'Cloudy'
        df.loc[df['clearness_index'] > 0.35, 'sky_regime'] = 'Clear'

        df.to_csv(filename, index=False, encoding='utf-8')
        print(f'Saved: {filename} ({len(df):,} rows)')
        return True

    except requests.exceptions.RequestException as exc:
        print(f'Request failed: {exc}')
        return False
    except Exception as exc:
        print(f'Processing failed: {exc}')
        return False

## 8. Download all 25 site-year datasets

This cell replaces the original `main()` function. It creates the checkpoint state, skips completed site-year combinations, downloads pending combinations, saves the checkpoint after each successful file, and waits 1.1 seconds between requests.

In [9]:
completed = load_checkpoint()

total = len(SITES) * len(YEARS)
success_count = 0
fail_count = 0
skip_count = 0

for site in SITES:
    print(f'\n--- {site["name"]} ({site["climate"]}) ---')

    for year in YEARS:
        key = (site['name'], year)
        filename = os.path.join(OUTPUT_DIR, f"{site['name']}_{year}.csv")

        if key in completed and os.path.exists(filename):
            print(f'[skip] {site["name"]} {year} already downloaded')
            skip_count += 1
            continue

        ok = download_site_year(site, year)

        if ok:
            completed.add(key)
            save_checkpoint(completed)
            success_count += 1
        else:
            fail_count += 1

        time.sleep(1.1)

print('\n=== Download complete ===')
print(f'Success : {success_count}')
print(f'Skipped : {skip_count}')
print(f'Failed  : {fail_count}')
print(f'Completed: {len(completed)} / {total}')


--- Phoenix_AZ (hot_desert) ---
Saved: nsrdb_data/Phoenix_AZ_2018.csv (8,760 rows)
Saved: nsrdb_data/Phoenix_AZ_2019.csv (8,760 rows)
Saved: nsrdb_data/Phoenix_AZ_2020.csv (8,784 rows)
Saved: nsrdb_data/Phoenix_AZ_2021.csv (8,760 rows)
Saved: nsrdb_data/Phoenix_AZ_2022.csv (8,760 rows)

--- LosAngeles_CA (mediterranean) ---
Saved: nsrdb_data/LosAngeles_CA_2018.csv (8,760 rows)
Saved: nsrdb_data/LosAngeles_CA_2019.csv (8,760 rows)
Saved: nsrdb_data/LosAngeles_CA_2020.csv (8,784 rows)
Saved: nsrdb_data/LosAngeles_CA_2021.csv (8,760 rows)
Saved: nsrdb_data/LosAngeles_CA_2022.csv (8,760 rows)

--- Denver_CO (semi_arid) ---
Saved: nsrdb_data/Denver_CO_2018.csv (8,760 rows)
Saved: nsrdb_data/Denver_CO_2019.csv (8,760 rows)
Saved: nsrdb_data/Denver_CO_2020.csv (8,784 rows)
Saved: nsrdb_data/Denver_CO_2021.csv (8,760 rows)
Saved: nsrdb_data/Denver_CO_2022.csv (8,760 rows)

--- Miami_FL (subtropical) ---
Saved: nsrdb_data/Miami_FL_2018.csv (8,760 rows)
Saved: nsrdb_data/Miami_FL_2019.csv (8,76

## 9. Verify the downloaded files

A complete run should have 25 completed site-year combinations and 25 CSV files in `nsrdb_data/`. If some downloads failed, re-run the download cell; completed combinations will be skipped.

In [10]:
completed = load_checkpoint()
expected = len(SITES) * len(YEARS)

csv_files = sorted(
    f for f in os.listdir(OUTPUT_DIR)
    if f.lower().endswith('.csv')
)

print(f'Checkpoint entries: {len(completed)} / {expected}')
print(f'CSV files found  : {len(csv_files)} / {expected}')

for filename in csv_files:
    print(f'  {filename}')

if len(completed) == expected and len(csv_files) == expected:
    print('\nAll expected files are available.')
    print('Next step: run merge_nsrdb.py')
else:
    print('\nDownload is incomplete. Re-run the download cell after checking API errors.')

Checkpoint loaded: 25 completed files.
Checkpoint entries: 25 / 25
CSV files found  : 25 / 25
  Denver_CO_2018.csv
  Denver_CO_2019.csv
  Denver_CO_2020.csv
  Denver_CO_2021.csv
  Denver_CO_2022.csv
  LosAngeles_CA_2018.csv
  LosAngeles_CA_2019.csv
  LosAngeles_CA_2020.csv
  LosAngeles_CA_2021.csv
  LosAngeles_CA_2022.csv
  Miami_FL_2018.csv
  Miami_FL_2019.csv
  Miami_FL_2020.csv
  Miami_FL_2021.csv
  Miami_FL_2022.csv
  Phoenix_AZ_2018.csv
  Phoenix_AZ_2019.csv
  Phoenix_AZ_2020.csv
  Phoenix_AZ_2021.csv
  Phoenix_AZ_2022.csv
  Seattle_WA_2018.csv
  Seattle_WA_2019.csv
  Seattle_WA_2020.csv
  Seattle_WA_2021.csv
  Seattle_WA_2022.csv

All expected files are available.
Next step: run merge_nsrdb.py
